# 07 Trade Review and Attribution

本课把完整交易日志继续拆解，观察每笔交易对最终净值的贡献。

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from quant_trading.attribution import (
    add_trade_attribution,
    plot_trade_attribution,
    select_top_contributors,
    summarize_by_duration,
    summarize_trade_attribution,
)
from quant_trading.execution import build_round_trip_trade_log
from quant_trading.market_data import download_ohlcv
from quant_trading.moving_average import add_moving_average_strategy_next_open

## 1. 生成交易日志

沿用第 6 课的 SPY 10/200 双均线策略。

In [ ]:
df = download_ohlcv(symbol="SPY", start="2000-01-01", auto_adjust=True)
strategy = add_moving_average_strategy_next_open(
    df,
    short_window=10,
    long_window=200,
    transaction_cost_bps=0.0,
)
trade_log = build_round_trip_trade_log(
    strategy,
    slippage_bps=2.0,
    commission_bps=1.0,
)
trade_log.head()

## 2. 增加收益归因字段

`equity_gain` 衡量这笔交易让累计净值增加或减少了多少。

In [ ]:
attributed = add_trade_attribution(trade_log)
attributed[[
    "trade_number",
    "entry_date",
    "exit_date",
    "net_return",
    "starting_equity",
    "ending_equity",
    "equity_gain",
    "contribution_to_total_gain",
]].head()

## 3. 查看收益集中度

In [ ]:
summary = summarize_trade_attribution(attributed)
summary

## 4. 前 5 大贡献和最差 5 笔

In [ ]:
select_top_contributors(attributed, n=5, largest=True)[[
    "trade_number", "entry_date", "exit_date", "holding_days",
    "net_return", "equity_gain", "contribution_to_total_gain",
]]

In [ ]:
select_top_contributors(attributed, n=5, largest=False)[[
    "trade_number", "entry_date", "exit_date", "holding_days",
    "net_return", "equity_gain", "contribution_to_total_gain",
]]

## 5. 持有周期归因

In [ ]:
duration_summary = summarize_by_duration(attributed)
duration_summary

## 6. 画归因图

In [ ]:
plot_trade_attribution(attributed, duration_summary);